In [1]:
import sys
print(sys.executable)
import os
print(os.chdir(".."))
print(os.getcwd())

c:\Users\Admin\Projects\credit_risk_score\credit-risk-scoring\venv312\Scripts\python.exe
None
c:\Users\Admin\Projects\credit_risk_score\credit-risk-scoring


In [2]:
import joblib
import pandas as pd
import numpy as np

X_fe = joblib.load("data/X_fe.joblib")
y = joblib.load("data/y.joblib")
imputer = joblib.load("data/imputer_fe.joblib")
scaler = joblib.load("data/scaler_fe.joblib")
log_reg2 = joblib.load("data/log_reg_fe.joblib")

In [3]:
coef_df = pd.DataFrame({
    "feature": X_fe.columns,
    "coefficient": log_reg2.coef_[0]
}).sort_values(by="coefficient", ascending=False)

coef_df["odds_ratio"] = np.exp(coef_df["coefficient"])
coef_df


,feature,coefficient,odds_ratio
1,credit_utilization,1.032911,2.809230
7,"months_(36, 72]",0.926387,2.525369
0,num_past_dues,0.761104,2.140638
4,"age_(45, 60]",0.396550,1.486687
8,"months_(72, 300]",0.377990,1.459348
2,income_missing,0.264204,1.302393
3,"age_(30, 45]",0.244199,1.276598
9,income_log,0.112016,1.118531
5,"age_(60, 100]",0.083831,1.087445
6,"months_(12, 36]",-0.492372,0.611175


In [4]:
example_raw = X_fe.iloc[[0]]

example_imp = imputer.transform(example_raw)
example_scaled = scaler.transform(example_imp)


In [5]:
contrib_df = pd.DataFrame({
    "feature": X_fe.columns,
    "value_scaled": example_scaled[0],
    "coefficient": log_reg2.coef_[0]
})

contrib_df["contribution"] = (
    contrib_df["value_scaled"] * contrib_df["coefficient"]
)

contrib_df = contrib_df.sort_values(by="contribution", ascending=False)
contrib_df

,feature,value_scaled,coefficient,contribution
3,"age_(30, 45]",1.128152,0.244199,0.275494
5,"age_(60, 100]",-0.294884,0.083831,-0.024720
9,income_log,-0.291409,0.112016,-0.032642
2,income_missing,-0.294884,0.264204,-0.077909
8,"months_(72, 300]",-0.369274,0.377990,-0.139582
4,"age_(45, 60]",-0.369274,0.396550,-0.146436
0,num_past_dues,-0.658553,0.761104,-0.501227
7,"months_(36, 72]",-0.561951,0.926387,-0.520585
6,"months_(12, 36]",1.333333,-0.492372,-0.656496
1,credit_utilization,-0.817122,1.032911,-0.844014


In [6]:
risk_reasons = contrib_df[contrib_df["contribution"] > 0].head(3)
risk_reasons

,feature,value_scaled,coefficient,contribution
3,"age_(30, 45]",1.128152,0.244199,0.275494


## Model Explainability

We used logistic regression, which allows direct interpretation through feature coefficients.

Each prediction is decomposed into per-feature contributions:

Contribution = (standardized feature value) × (model coefficient)

- Positive contribution → increases default risk  
- Negative contribution → decreases default risk  

All computations are performed in the scaled feature space to maintain consistency with model training.

---

## Key Observations

- Credit utilization and past delinquencies are the strongest global risk drivers.
- Time-on-book (loan duration) increases risk due to exposure.
- Age shows non-monotonic behavior, handled via binning.
- Income has limited effect after transformation.

---

## Risk Reason Codes (Example Loan)

For the selected loan:

Top contributing risk factor:
- Age group (30–45)

All other features contributed negatively, indicating a relatively low-risk profile overall.

---

## Notes on Scaling

Standard scaling was applied to all features for simplicity.

In production systems:
- Continuous variables are scaled
- Binary/dummy variables are typically left unscaled

This does not affect model validity but slightly changes interpretation scale.

---

## Why Logistic Regression

- Fully interpretable
- Stable under small datasets
- Suitable for regulated environments like credit risk

More complex models (e.g., XGBoost) require additional tools like SHAP for explainability.

## Note on Reproducibility

This notebook uses pre-trained artifacts (model, scaler, imputer) generated in the EDA pipeline.

To reproduce results:
1. Run notebooks/EDA.ipynb
2. This will generate artifacts in the data/ folder
3. Then run this notebook

Artifacts are not committed to GitHub to keep the repository clean and reproducible.